In [ ]:
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Constants
ATTACK_JSON_PATH = Path("../data/attck/enterprise-attack.json")
RAW_CICIDS_DIR   = Path("../data/cicids")
INPUT_CSV_PATH   = Path("../data/processed/cicids_processed.csv")
OUTPUT_CSV_PATH  = Path("../data/processed/cicids_relabelled_semantic.csv")

In [ ]:
# functions to clean and normalize text
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).lower()
    text = text.replace("-", " ")
    text = re.sub(r"[^a-z0-9\.\s/_]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def safe_number(x, default=0.0):
    try:
        if x is None:
            return default
        if isinstance(x, float) and math.isnan(x):
            return default
        return float(x)
    except Exception:
        return default


def clean_label_text(x: str) -> str:
    x = str(x).strip()
    x = x.replace("�", "-")
    x = x.replace("–", "-")
    x = x.replace("—", "-")
    x = re.sub(r"\s*-\s*", " - ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

In [ ]:
# Load raw CICIDS data
raw_files = sorted(RAW_CICIDS_DIR.glob("*.csv"))

print("Found raw CICIDS files:", len(raw_files))
for f in raw_files:
    print("-", f.name)

dfs = []
for f in raw_files:
    temp = pd.read_csv(f)
    temp["source_file"] = f.name
    dfs.append(temp)

raw_df = pd.concat(dfs, ignore_index=True)

print("Combined raw shape:", raw_df.shape)
raw_df.head(2)

Found raw CICIDS files: 8
- Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
- Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
- Friday-WorkingHours-Morning.pcap_ISCX.csv
- Monday-WorkingHours.pcap_ISCX.csv
- Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
- Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
- Tuesday-WorkingHours.pcap_ISCX.csv
- Wednesday-workingHours.pcap_ISCX.csv
Combined raw shape: (2830743, 80)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [ ]:
# Standardize and clean the raw CICIDS dataframe
def standardize_cicids_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "Destination Port": "dst_port",
        "Dst Port": "dst_port",
        "Protocol": "protocol",
        "Flow Duration": "flow_duration",
        "Total Fwd Packets": "tot_fwd_pkts",
        "Tot Fwd Pkts": "tot_fwd_pkts",
        "Total Backward Packets": "tot_bwd_pkts",
        "Tot Bwd Pkts": "tot_bwd_pkts",
        "Flow Bytes/s": "flow_bytes_s",
        "Flow Byts/s": "flow_bytes_s",
        "Flow Packets/s": "flow_packets_s",
        "Flow Pkts/s": "flow_packets_s",
        "SYN Flag Count": "syn_flag_count",
        "RST Flag Count": "rst_flag_count",
        "ACK Flag Count": "ack_flag_count",
        "PSH Flag Count": "psh_flag_count",
        "Label": "label",
    }

    for old, new in rename_map.items():
        if old in df.columns:
            df = df.rename(columns={old: new})

    wanted_cols = [
        "dst_port",
        "protocol",
        "flow_duration",
        "tot_fwd_pkts",
        "tot_bwd_pkts",
        "flow_bytes_s",
        "flow_packets_s",
        "syn_flag_count",
        "rst_flag_count",
        "ack_flag_count",
        "psh_flag_count",
        "label",
        "source_file",
    ]

    existing = [c for c in wanted_cols if c in df.columns]
    proc = df[existing].copy()

    if "label" in proc.columns:
        proc["label"] = proc["label"].astype(str).str.strip()
        proc["label"] = proc["label"].apply(clean_label_text)

    numeric_cols = [c for c in proc.columns if c not in ["label", "source_file"]]
    for c in numeric_cols:
        proc[c] = pd.to_numeric(proc[c], errors="coerce")

    return proc


proc_df = standardize_cicids_dataframe(raw_df)

# Exclude Monday (benign-only day)
proc_df = proc_df[proc_df["source_file"] != "Monday-WorkingHours.pcap_ISCX.csv"].copy()

print("After excluding Monday:", proc_df.shape)
print(proc_df["label"].value_counts())

print("Processed shape:", proc_df.shape)
print(proc_df.columns.tolist())
proc_df.head(3)

After excluding Monday: (2300825, 12)
label
BENIGN                        1743179
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP - Patator                    7938
SSH - Patator                    5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
Processed shape: (2300825, 12)
['dst_port', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts', 'flow_bytes_s', 'flow_packets_s', 'syn_flag_count', 'rst_flag_count', 'ack_flag_count', 'psh_flag_count', 'label', 'source_file']


,dst_port,flow_duration,tot_fwd_pkts,tot_bwd_pkts,flow_bytes_s,flow_packets_s,syn_flag_count,rst_flag_count,ack_flag_count,psh_flag_count,label,source_file
0,54865,3,2,0,4.000000e+06,666666.66670,0,0,1,0,BENIGN,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
1,55054,109,1,1,1.100917e+05,18348.62385,0,0,1,0,BENIGN,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2,55055,52,1,1,2.307692e+05,38461.53846,0,0,1,0,BENIGN,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [ ]:
# Stratified sampling
RANDOM_STATE = 42

def stratified_cap_sample(df, label_col="label", benign_cap=3000, attack_cap=2000):
    sampled_parts = []

    for label, group in df.groupby(label_col, dropna=False):
        n = len(group)

        if str(label).strip().upper() == "BENIGN":
            cap = benign_cap
        else:
            cap = attack_cap

        if n <= cap:
            sampled_parts.append(group)
        else:
            sampled_parts.append(group.sample(n=cap, random_state=RANDOM_STATE))

    sampled_df = pd.concat(sampled_parts, ignore_index=True)
    return sampled_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


sampled_df = stratified_cap_sample(
    proc_df,
    label_col="label",
    benign_cap=3000,
    attack_cap=2000
)

print("Sampled shape:", sampled_df.shape)
print(sampled_df["label"].value_counts())

Sampled shape: (23193, 12)
label
BENIGN                        3000
FTP - Patator                 2000
PortScan                      2000
DoS slowloris                 2000
SSH - Patator                 2000
DDoS                          2000
DoS Slowhttptest              2000
DoS Hulk                      2000
DoS GoldenEye                 2000
Bot                           1966
Web Attack - Brute Force      1507
Web Attack - XSS               652
Infiltration                    36
Web Attack - Sql Injection      21
Heartbleed                      11
Name: count, dtype: int64


In [ ]:
# Save the sampled processed CSV
INPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
sampled_df.to_csv(INPUT_CSV_PATH, index=False)
print("Saved sampled processed CSV to:", INPUT_CSV_PATH.resolve())

Saved sampled processed CSV to: /home/amenadiel/RAG-Based-Incident-Reporting-on-SIEM-Log-Streams/data/processed/cicids_processed.csv


In [ ]:
# Load the sampled processed CSV to verify
df = pd.read_csv(INPUT_CSV_PATH)
print(df.shape)
print(df["label"].value_counts())

(23193, 12)
label
BENIGN                        3000
FTP - Patator                 2000
PortScan                      2000
DoS slowloris                 2000
SSH - Patator                 2000
DDoS                          2000
DoS Slowhttptest              2000
DoS Hulk                      2000
DoS GoldenEye                 2000
Bot                           1966
Web Attack - Brute Force      1507
Web Attack - XSS               652
Infiltration                    36
Web Attack - Sql Injection      21
Heartbleed                      11
Name: count, dtype: int64


In [ ]:
# Load ATT&CK data
with open(ATTACK_JSON_PATH, "r", encoding="utf-8") as f:
    attack_data = json.load(f)

print("Loaded ATT&CK objects:", len(attack_data.get("objects", [])))

Loaded ATT&CK objects: 24771


In [ ]:
# Build protocol and port maps
PROTOCOL_MAP = {
    1: "icmp",
    6: "tcp",
    17: "udp"
}

PORT_SERVICE_MAP = {
    20: "ftp-data",
    21: "ftp",
    22: "ssh",
    23: "telnet",
    25: "smtp",
    53: "dns",
    67: "dhcp",
    68: "dhcp",
    69: "tftp",
    80: "http",
    88: "kerberos",
    110: "pop3",
    123: "ntp",
    135: "rpc",
    137: "netbios",
    138: "netbios",
    139: "smb",
    143: "imap",
    161: "snmp",
    389: "ldap",
    443: "https",
    445: "smb",
    514: "syslog",
    993: "imaps",
    995: "pop3s",
    1433: "mssql",
    1521: "oracle",
    3306: "mysql",
    3389: "rdp",
    5432: "postgresql",
    5900: "vnc",
    6379: "redis",
    8080: "http-alt",
    8443: "https-alt",
}

# functions to infer protocol and service names
def infer_protocol(protocol_value):
    try:
        p = int(float(protocol_value))
        return PROTOCOL_MAP.get(p, f"proto_{p}")
    except Exception:
        return "unknown"


def infer_service(dst_port):
    try:
        port = int(float(dst_port))
        return PORT_SERVICE_MAP.get(port, "unknown")
    except Exception:
        return "unknown"

# functions to categorize numeric features
def categorize_duration(duration_seconds):
    if duration_seconds <= 0.1:
        return "very_short"
    elif duration_seconds <= 1.0:
        return "short"
    elif duration_seconds <= 10.0:
        return "medium"
    return "long"


def categorize_packet_rate(flow_packets_s):
    if flow_packets_s > 10000:
        return "extreme"
    elif flow_packets_s > 1000:
        return "high"
    elif flow_packets_s > 100:
        return "moderate"
    return "low"

In [ ]:
# Building a text corpus from ATT&CK techniques for semantic similarity
def build_attack_text_corpus(attack_data):
    corpus = []

    for obj in attack_data.get("objects", []):
        if obj.get("type") != "attack-pattern":
            continue
        if obj.get("revoked", False) or obj.get("x_mitre_deprecated", False):
            continue

        attack_id = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                attack_id = ref.get("external_id")
                break

        if not attack_id:
            continue

        name = obj.get("name", "")
        desc = obj.get("description", "")
        stix_object_id = obj.get("id", "")
        phases = [p.get("phase_name", "") for p in obj.get("kill_chain_phases", [])]
        platforms = obj.get("x_mitre_platforms", [])
        data_sources = obj.get("x_mitre_data_sources", [])

        searchable_text = " ".join([
            name,
            desc,
            " ".join(phases),
            " ".join(platforms) if platforms else "",
            " ".join(data_sources) if data_sources else ""
        ]).strip()

        corpus.append({
            "stix_object_id": stix_object_id,
            "technique_id": attack_id,
            "technique_name": name,
            "description": desc,
            "kill_chain_phases": phases,
            "platforms": platforms,
            "data_sources": data_sources,
            "text": searchable_text,
        })

    return corpus


attack_corpus = build_attack_text_corpus(attack_data)

print("Technique count:", len(attack_corpus))
print(attack_corpus[0]["technique_id"], "-", attack_corpus[0]["technique_name"])

Technique count: 691
T1055.011 - Extra Window Memory Injection


In [ ]:
# Encode the ATT&CK technique texts using SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

attack_texts = [a["text"] for a in attack_corpus]
attack_embeddings = model.encode(
    attack_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", attack_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Embeddings shape: (691, 384)


In [ ]:
# Function to derive behavior profile from a CICIDS row
def derive_behavior_profile(row):
    label = str(row.get("label", "")).strip()
    label_norm = normalize_text(label)

    dst_port = row.get("dst_port", np.nan)
    protocol = infer_protocol(row.get("protocol", np.nan))

    duration_raw = safe_number(row.get("flow_duration", 0))
    duration_seconds = duration_raw / 1e6

    tot_fwd_pkts = safe_number(row.get("tot_fwd_pkts", 0))
    tot_bwd_pkts = safe_number(row.get("tot_bwd_pkts", 0))
    flow_packets_s = safe_number(row.get("flow_packets_s", 0))
    flow_bytes_s = safe_number(row.get("flow_bytes_s", 0))

    syn_flag = safe_number(row.get("syn_flag_count", 0))
    rst_flag = safe_number(row.get("rst_flag_count", 0))
    ack_flag = safe_number(row.get("ack_flag_count", 0))
    psh_flag = safe_number(row.get("psh_flag_count", 0))

    service_target = infer_service(dst_port)
    duration_class = categorize_duration(duration_seconds)
    packet_rate_class = categorize_packet_rate(flow_packets_s)

    if label_norm == "benign":
        return {
            "raw_label": label,
            "behavior_type": "benign",
            "service_target": service_target,
            "protocol_context": protocol,
            "duration_seconds": round(duration_seconds, 9),
            "duration_class": duration_class,
            "packet_rate_class": packet_rate_class,
            "flow_packets_s": round(flow_packets_s, 3),
            "flow_bytes_s": round(flow_bytes_s, 3),
            "tot_fwd_pkts": round(tot_fwd_pkts, 3),
            "tot_bwd_pkts": round(tot_bwd_pkts, 3),
            "traffic_pattern": [],
            "evidence_tokens": [],
        }

    behavior_type = "unknown"
    evidence_tokens = []
    traffic_pattern = []

    if "portscan" in label_norm or "port scan" in label_norm:
        behavior_type = "scan"
        evidence_tokens += ["port_scan", "service_discovery", "many_targets"]
        traffic_pattern += ["short_connections", "enumeration_like"]

    elif "ssh patator" in label_norm:
        behavior_type = "brute_force"
        evidence_tokens += ["ssh", "repeated_auth_attempts", "password_guessing"]
        traffic_pattern += ["authentication_targeting", "repetition"]

    elif "ftp patator" in label_norm:
        behavior_type = "brute_force"
        evidence_tokens += ["ftp", "repeated_auth_attempts", "password_guessing"]
        traffic_pattern += ["authentication_targeting", "repetition"]

    elif "web attack brute force" in label_norm or "brute force" in label_norm:
        behavior_type = "brute_force"
        evidence_tokens += ["http", "repeated_auth_attempts", "password_guessing", "web_login_abuse"]
        traffic_pattern += ["authentication_targeting", "application_layer_attack", "repetition"]

    elif "ddos" in label_norm:
        behavior_type = "flooding"
        evidence_tokens += ["distributed_denial_of_service", "traffic_flood", "resource_exhaustion"]
        traffic_pattern += ["availability_impact", "flooding_behavior"]

    elif "dos" in label_norm:
        behavior_type = "flooding"
        evidence_tokens += ["denial_of_service", "traffic_flood", "resource_exhaustion"]
        traffic_pattern += ["availability_impact", "flooding_behavior"]

    elif "sql injection" in label_norm:
        behavior_type = "exploit"
        evidence_tokens += ["sql_injection", "web_exploit", "application_abuse"]
        traffic_pattern += ["application_layer_attack"]

    elif "xss" in label_norm:
        behavior_type = "exploit"
        evidence_tokens += ["cross_site_scripting", "xss", "web_exploit"]
        traffic_pattern += ["application_layer_attack"]

    elif "heartbleed" in label_norm:
        behavior_type = "exploit"
        evidence_tokens += ["heartbleed", "tls_exploit", "public_facing_service"]
        traffic_pattern += ["application_layer_attack"]

    elif "bot" in label_norm:
        behavior_type = "c2_or_bot"
        evidence_tokens += ["bot_activity", "command_and_control"]
        traffic_pattern += ["persistent_or_repeated_comm"]

    elif "infiltration" in label_norm or "infilteration" in label_norm:
        behavior_type = "post_compromise"
        evidence_tokens += ["malicious_file", "internal_reconnaissance", "post_compromise"]
        traffic_pattern += ["multi_stage_possible"]

    if syn_flag > 0 and ack_flag == 0:
        traffic_pattern.append("syn_heavy")
    if rst_flag > 0:
        traffic_pattern.append("connection_reset")
    if psh_flag > 0:
        traffic_pattern.append("push_data")
    if flow_packets_s > 1000:
        traffic_pattern.append("high_rate_packets")
    if flow_bytes_s > 1e6:
        traffic_pattern.append("high_volume_bytes")

    return {
        "raw_label": label,
        "behavior_type": behavior_type,
        "service_target": service_target,
        "protocol_context": protocol,
        "duration_seconds": round(duration_seconds, 9),
        "duration_class": duration_class,
        "packet_rate_class": packet_rate_class,
        "flow_packets_s": round(flow_packets_s, 3),
        "flow_bytes_s": round(flow_bytes_s, 3),
        "tot_fwd_pkts": round(tot_fwd_pkts, 3),
        "tot_bwd_pkts": round(tot_bwd_pkts, 3),
        "traffic_pattern": sorted(set(traffic_pattern)),
        "evidence_tokens": sorted(set(evidence_tokens)),
    }

In [ ]:
# Function to build a descriptive text for the attack profile
def build_attack_description(profile):
    patterns = ", ".join(profile["traffic_pattern"]) if profile["traffic_pattern"] else "none"
    evidence = ", ".join(profile["evidence_tokens"]) if profile["evidence_tokens"] else "none"

    text = (
        f"This network activity is labeled as {profile['raw_label']}. "
        f"It shows {profile['behavior_type']} behavior. "
        f"It targets {profile['service_target']} services over {profile['protocol_context']}. "
        f"Observed duration is {profile['duration_class']} "
        f"({profile['duration_seconds']} seconds). "
        f"Packet rate is {profile['packet_rate_class']} with "
        f"{profile['flow_packets_s']} flow packets per second and "
        f"{profile['flow_bytes_s']} flow bytes per second. "
        f"Observed traffic patterns include: {patterns}. "
        f"Evidence indicators include: {evidence}. "
    )

    if profile["behavior_type"] == "scan":
        text += "This suggests network service scanning, reconnaissance, and discovery activity."
    elif profile["behavior_type"] == "brute_force":
        text += "This suggests repeated authentication attempts, password guessing, and brute force activity."
    elif profile["behavior_type"] == "flooding":
        text += "This suggests denial of service, flooding, and service or network exhaustion activity."
    elif profile["behavior_type"] == "exploit":
        text += "This suggests exploitation of a public-facing application or an application-layer attack."
    elif profile["behavior_type"] == "c2_or_bot":
        text += "This suggests bot-related or command-and-control behavior."
    elif profile["behavior_type"] == "post_compromise":
        text += "This suggests staged or post-compromise activity, possibly including malicious execution and discovery."

    return text

In [ ]:
# Function to compute a heuristic cue matching score between the behavior profile and an ATT&CK technique
def cue_match_score(profile, attack_obj):
    score = 0.0

    text = normalize_text(
        attack_obj["technique_name"] + " " +
        attack_obj["description"] + " " +
        " ".join(attack_obj.get("kill_chain_phases", []))
    )

    behavior = profile["behavior_type"]
    label = normalize_text(profile["raw_label"])
    service = normalize_text(profile["service_target"])

    # SCAN
    if behavior == "scan":
        if "network service discovery" in text:
            score += 8.0
        if "scanning ip blocks" in text:
            score += 2.0
        if "scan" in text or "scanning" in text:
            score += 1.5
        if "discovery" in text:
            score += 0.8

    # BRUTE FORCE
    if behavior == "brute_force":
        if "brute force" in text:
            score += 8.0
        if "password guessing" in text:
            score += 7.0
        if "password spraying" in text:
            score += 3.0
        if "credential" in text:
            score += 1.5

        if "ssh" in label and "ssh" in text:
            score += 2.0
        if "ftp" in label and "ftp" in text:
            score += 2.0
        if "web" in label and "http" in service:
            score += 1.5

    # FLOODING
    if behavior == "flooding":
        if "network denial of service" in text:
            score += 8.0
        if "direct network flood" in text:
            score += 4.0
        if "service exhaustion flood" in text:
            score += 3.0
        if "application exhaustion flood" in text:
            score += 2.5
        if "denial of service" in text:
            score += 1.5

    # EXPLOIT
    if behavior == "exploit":
        if "exploit public facing application" in text:
            score += 10.0
        if "initial access" in text:
            score += 1.5

        if "sql injection" in label and "exploit public facing application" in text:
            score += 6.0

        if "xss" in label and "exploit public facing application" in text:
            score += 4.0

        # penalize clearly wrong exploit matches
        if "denial of service" in text:
            score -= 3.0
        if "web protocols" in text:
            score -= 1.0

    # BOT / C2
    if behavior == "c2_or_bot":
        if "command and control" in text:
            score += 4.0
        if "botnet" in text:
            score += 3.0
        if "application layer protocol" in text:
            score += 2.0

    # POST-COMPROMISE
    if behavior == "post_compromise":
        if "user execution" in text:
            score += 3.0
        if "malicious file" in text:
            score += 3.0
        if "network service discovery" in text:
            score += 1.5
        if "discovery" in text:
            score += 1.0

    return score

In [ ]:
# Main function to retrieve top ATT&CK technique candidates based on combined embedding similarity and heuristic cue matching
def retrieve_attack_candidates_embedding(profile, top_k=5, alpha=0.55, beta=0.45):
    if profile["behavior_type"] == "benign":
        desc = (
            f"This network activity is labeled as {profile['raw_label']}. "
            f"It is benign traffic and is excluded from ATT&CK technique mapping."
        )
        return desc, []

    desc = build_attack_description(profile)
    desc_emb = model.encode([desc], normalize_embeddings=True)

    sims = cosine_similarity(desc_emb, attack_embeddings)[0]

    candidates = []
    for idx, attack_obj in enumerate(attack_corpus):
        emb_score = float(sims[idx])
        rule_score = cue_match_score(profile, attack_obj)
        final_score = alpha * emb_score + beta * rule_score

        candidates.append({
            "stix_object_id": attack_obj["stix_object_id"],
            "technique_id": attack_obj["technique_id"],
            "technique_name": attack_obj["technique_name"],
            "embedding_score": emb_score,
            "rule_score": rule_score,
            "score": final_score,
            "description": attack_obj["description"],
            "kill_chain_phases": attack_obj["kill_chain_phases"],
        })

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)[:top_k]
    return desc, candidates


def assign_from_candidates(candidates):
    if not candidates:
        return None, None, "none"

    score = candidates[0]["score"]
    if score >= 0.60:
        conf = "high"
    elif score >= 0.40:
        conf = "medium"
    else:
        conf = "low"

    return candidates[0]["technique_id"], candidates[0]["technique_name"], conf

In [ ]:
# Function to explain the mapping for a given label value
def explain_mapping_for_label(label_value, df, top_k=5):
    subset = df[df["label"] == label_value]
    if subset.empty:
        print(f"No rows found for label: {label_value}")
        return

    row = subset.iloc[0].to_dict()
    profile = derive_behavior_profile(row)
    desc, candidates = retrieve_attack_candidates_embedding(profile, top_k=top_k)

    print("=" * 100)
    print("CIC-IDS LABEL")
    print(label_value)

    print("\nDERIVED BEHAVIOR PROFILE")
    print(json.dumps(profile, indent=2))

    print("\nATTACK DESCRIPTION USED FOR STIX LOOKUP")
    print(desc)

    print("\nTOP STIX ATT&CK CANDIDATES")
    for i, c in enumerate(candidates, 1):
        print(f"\nCandidate {i}")
        print("  STIX object id :", c["stix_object_id"])
        print("  ATT&CK id      :", c["technique_id"])
        print("  Technique name :", c["technique_name"])
        print("  Embedding score:", round(c.get("embedding_score", 0.0), 6))
        print("  Rule score     :", round(c.get("rule_score", 0.0), 6))
        print("  Final score    :", round(c["score"], 6))
        print("  Kill chain     :", c["kill_chain_phases"])
        print("  Description    :", c["description"][:250].replace("\n", " "), "...")

    if candidates:
        top = candidates[0]
        print("\nSELECTED TECHNIQUE")
        print(f"{top['technique_id']} - {top['technique_name']}")
        print("Reason: selected by highest combined score from semantic similarity and rule-based ATT&CK cue matching.")
    else:
        print("\nSELECTED TECHNIQUE")
        print("None")
        print("Reason: benign traffic is excluded from ATT&CK assignment.")

    print("=" * 100)

In [ ]:
# Example usage
sample_row = df.iloc[0].to_dict()

profile = derive_behavior_profile(sample_row)
desc, candidates = retrieve_attack_candidates_embedding(profile, top_k=5)
assigned_id, assigned_name, conf = assign_from_candidates(candidates)

print("PROFILE:")
print(json.dumps(profile, indent=2))

print("\nDESCRIPTION:")
print(desc)

print("\nTOP CANDIDATES:")
for c in candidates:
    print({
        "technique_id": c["technique_id"],
        "technique_name": c["technique_name"],
        "score": round(c["score"], 6)
    })

print("\nASSIGNED:", assigned_id, assigned_name, conf)

PROFILE:
{
  "raw_label": "Bot",
  "behavior_type": "c2_or_bot",
  "service_target": "unknown",
  "protocol_context": "unknown",
  "duration_seconds": 7.9e-05,
  "duration_class": "very_short",
  "packet_rate_class": "extreme",
  "flow_packets_s": 25316.456,
  "flow_bytes_s": 151898.734,
  "tot_fwd_pkts": 1.0,
  "tot_bwd_pkts": 1.0,
  "traffic_pattern": [
    "high_rate_packets",
    "persistent_or_repeated_comm"
  ],
  "evidence_tokens": [
    "bot_activity",
    "command_and_control"
  ]
}

DESCRIPTION:
This network activity is labeled as Bot. It shows c2_or_bot behavior. It targets unknown services over unknown. Observed duration is very_short (7.9e-05 seconds). Packet rate is extreme with 25316.456 flow packets per second and 151898.734 flow bytes per second. Observed traffic patterns include: high_rate_packets, persistent_or_repeated_comm. Evidence indicators include: bot_activity, command_and_control. This suggests bot-related or command-and-control behavior.

TOP CANDIDATES:
{'t

In [21]:
print(sorted(df["label"].dropna().unique()))

['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP - Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH - Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [22]:
explain_mapping_for_label("BENIGN", df)
explain_mapping_for_label("PortScan", df)
explain_mapping_for_label("DDoS", df)
explain_mapping_for_label("SSH - Patator", df)
explain_mapping_for_label("FTP - Patator", df)
explain_mapping_for_label("Web Attack - Brute Force", df)
explain_mapping_for_label("Web Attack - XSS", df)
explain_mapping_for_label("Web Attack - Sql Injection", df)

CIC-IDS LABEL
BENIGN

DERIVED BEHAVIOR PROFILE
{
  "raw_label": "BENIGN",
  "behavior_type": "benign",
  "service_target": "https",
  "protocol_context": "unknown",
  "duration_seconds": 115.497875,
  "duration_class": "long",
  "packet_rate_class": "low",
  "flow_packets_s": 0.355,
  "flow_bytes_s": 69.516,
  "tot_fwd_pkts": 21.0,
  "tot_bwd_pkts": 20.0,
  "traffic_pattern": [],
  "evidence_tokens": []
}

ATTACK DESCRIPTION USED FOR STIX LOOKUP
This network activity is labeled as BENIGN. It is benign traffic and is excluded from ATT&CK technique mapping.

TOP STIX ATT&CK CANDIDATES

SELECTED TECHNIQUE
None
Reason: benign traffic is excluded from ATT&CK assignment.
CIC-IDS LABEL
PortScan

DERIVED BEHAVIOR PROFILE
{
  "raw_label": "PortScan",
  "behavior_type": "scan",
  "service_target": "unknown",
  "protocol_context": "unknown",
  "duration_seconds": 5.4e-05,
  "duration_class": "very_short",
  "packet_rate_class": "extreme",
  "flow_packets_s": 37037.037,
  "flow_bytes_s": 148148.14

In [23]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    row_dict = row.to_dict()
    profile = derive_behavior_profile(row_dict)
    desc, candidates = retrieve_attack_candidates_embedding(profile, top_k=5)
    assigned_id, assigned_name, conf = assign_from_candidates(candidates)

    records.append({
        "behavior_profile_json": json.dumps(profile),
        "attack_description": desc,
        "candidate_technique_ids_json": json.dumps([c["technique_id"] for c in candidates]),
        "candidate_technique_names_json": json.dumps([c["technique_name"] for c in candidates]),
        "candidate_scores_json": json.dumps([round(c["score"], 6) for c in candidates]),
        "candidate_stix_object_ids_json": json.dumps([c["stix_object_id"] for c in candidates]),
        "candidate_kill_chain_json": json.dumps([c["kill_chain_phases"] for c in candidates]),
        "candidate_attck_technique_id": assigned_id,
        "candidate_attck_technique_name": assigned_name,
        "mapping_confidence": conf,
        "mapping_method": "semantic_similarity_over_attack_pattern_stix_objects",
    })

mapping_df = pd.DataFrame(records)
df_relabelled = pd.concat([df.reset_index(drop=True), mapping_df.reset_index(drop=True)], axis=1)

print(df_relabelled.shape)
df_relabelled.head(3)

  0%|          | 0/23193 [00:00<?, ?it/s]

(23193, 23)


,dst_port,flow_duration,tot_fwd_pkts,tot_bwd_pkts,flow_bytes_s,flow_packets_s,syn_flag_count,rst_flag_count,ack_flag_count,psh_flag_count,...,attack_description,candidate_technique_ids_json,candidate_technique_names_json,candidate_scores_json,candidate_stix_object_ids_json,candidate_kill_chain_json,candidate_attck_technique_id,candidate_attck_technique_name,mapping_confidence,mapping_method
0,4294,79,1,1,151898.73420,25316.455700,0,0,1,0,...,This network activity is labeled as Bot. It sh...,"[""T1583.005"", ""T1584.008"", ""T1176.001"", ""T1071...","[""Botnet"", ""Network Devices"", ""Browser Extensi...","[3.38891, 3.306979, 3.225876, 2.933228, 2.906426]","[""attack-pattern--31225cd3-cd46-4575-b287-c2c1...","[[""resource-development""], [""resource-developm...",T1583.005,Botnet,high,semantic_similarity_over_attack_pattern_stix_o...
1,21,258,2,1,54263.56589,11627.906980,1,0,1,0,...,This network activity is labeled as FTP - Pata...,"[""T1110.001"", ""T1110"", ""T1595.003"", ""T1187"", ""...","[""Password Guessing"", ""Brute Force"", ""Wordlist...","[8.484974, 4.466402, 4.453855, 4.443174, 4.397...","[""attack-pattern--09c4c11e-4fa1-4f8c-8dad-3cf8...","[[""credential-access""], [""credential-access""],...",T1110.001,Password Guessing,high,semantic_similarity_over_attack_pattern_stix_o...
2,80,5598498,3,1,0.00000,0.714477,0,0,0,1,...,This network activity is labeled as Web Attack...,"[""T1189"", ""T1190"", ""T1595.002"", ""T1596.005"", ""...","[""Drive-by Compromise"", ""Exploit Public-Facing...","[7.230841, 7.19942, 7.181816, 7.167609, 7.152687]","[""attack-pattern--d742a578-d70e-4d0e-96a6-02a9...","[[""initial-access""], [""initial-access""], [""rec...",T1189,Drive-by Compromise,high,semantic_similarity_over_attack_pattern_stix_o...


In [24]:
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df_relabelled.to_csv(OUTPUT_CSV_PATH, index=False)

print("Saved relabelled CSV to:", OUTPUT_CSV_PATH.resolve())

Saved relabelled CSV to: /home/amenadiel/RAG-Based-Incident-Reporting-on-SIEM-Log-Streams/data/processed/cicids_relabelled_semantic.csv


In [25]:
summary = (
    df_relabelled.groupby(["label", "candidate_attck_technique_id", "candidate_attck_technique_name"])
    .size()
    .reset_index(name="count")
    .sort_values(["label", "count"], ascending=[True, False])
)

summary.head(50)

,label,candidate_attck_technique_id,candidate_attck_technique_name,count
0,Bot,T1583.005,Botnet,1966
1,DDoS,T1498,Network Denial of Service,2000
2,DoS GoldenEye,T1498,Network Denial of Service,2000
3,DoS Hulk,T1498,Network Denial of Service,2000
4,DoS Slowhttptest,T1498,Network Denial of Service,2000
5,DoS slowloris,T1498,Network Denial of Service,2000
6,FTP - Patator,T1110.001,Password Guessing,2000
7,Heartbleed,T1190,Exploit Public-Facing Application,11
8,Infiltration,T1608.001,Upload Malware,36
9,PortScan,T1046,Network Service Discovery,2000
